# Scope-Creep Retrospective — Three-Agent ML Pipeline

A three-agent OpenAI pipeline where the agents **disagree by design**:

- **Dr. Hong** (Team Lead) — writes the scope, reviews submitted code, trims out-of-scope additions, executes the final version.
- **Andrey Dermendzhiev** (Coder) — fulfills the scope then over-delivers. Extra models, cross-validation, SHAP, bonus plots. Does *not* execute.
- **Dimitar Dermendzhiev** (SCRUM Master) — builds a "lessons learned" deck, then re-opens the `.pptx` to run a structural QA loop. Regenerates up to 3 times if anything fails.

**Dataset:** [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) — 7,043 rows, binary `Churn` target. Downloaded automatically below.

Based on the multi-agent multiprocessing pattern from `5_heart_attack_analysis_v2.ipynb`, extended with scope-creep narrative, self-QA loops, structured event logging, and a live Rich terminal UI.

## Flow

```
Dr. Hong → Andrey (scope)
Andrey → Dr. Hong (inflated code, ~280 lines)
Dr. Hong reviews, trims (~40 lines), executes → prediction.csv
Dr. Hong → Dimitar (retrospective brief)
Dimitar generates deck, runs QA; regenerates if needed → presentation.pptx
```

## 1. Install dependencies

In [ ]:
!pip install -q openai pandas scikit-learn python-pptx rich

## 2. Download and split the dataset

Telco Churn → `training.csv` (with label) + `scoring.csv` (label stripped).

In [ ]:
"""Download Telco Customer Churn and split into training/scoring CSVs."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

TELCO_URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
)


def prepare_data(
    data_dir: str | Path = ".",
    test_fraction: float = 0.2,
    seed: int = 42,
    url: str = TELCO_URL,
) -> tuple[Path, Path]:
    """Download Telco Churn, split into training.csv and scoring.csv.

    Returns (training_path, scoring_path).
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(url)

    # TotalCharges comes in as object with blank strings — coerce
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

    # customerID is not a feature
    df = df.drop(columns=["customerID"])

    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_fraction))
    training = df.iloc[:split_idx].copy()
    scoring = df.iloc[split_idx:].drop(columns=["Churn"]).copy()

    training_path = data_dir / "training.csv"
    scoring_path = data_dir / "scoring.csv"
    training.to_csv(training_path, index=False)
    scoring.to_csv(scoring_path, index=False)

    return training_path, scoring_path


if __name__ == "__main__":
    tp, sp = prepare_data()
    print(f"Wrote {tp} and {sp}")


In [ ]:
prepare_data()
print('Data ready.')

## 3. Utilities — code extraction and transcripts

In [ ]:
"""Pull executable Python code out of LLM responses."""

from __future__ import annotations

import re

_PYTHON_FENCE = re.compile(r"```python\s*(.*?)```", re.DOTALL | re.IGNORECASE)
_ANY_FENCE = re.compile(r"```\s*(.*?)```", re.DOTALL)


def extract_python_code(text: str) -> str:
    """Return pure Python from a response that may wrap it in code fences.

    Handles three cases in order of preference:
    1. ```python ... ```  (explicit language tag)
    2. ```        ... ```  (generic fenced block)
    3. No fences — return the text as-is (best effort)
    """
    matches = _PYTHON_FENCE.findall(text)
    if matches:
        return "\n\n".join(m.strip() for m in matches)

    matches = _ANY_FENCE.findall(text)
    if matches:
        return "\n\n".join(m.strip() for m in matches)

    return text.strip()


def strip_code_fences(text: str) -> str:
    """Return everything OUTSIDE code fences — useful for extracting prose."""
    return re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()


In [ ]:
"""Save full agent transcripts to JSON for post-run inspection."""

from __future__ import annotations

import json
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path


@dataclass
class UIEvent:
    """A single visible event from an agent.

    These are what the live Rich UI renders and what gets persisted
    to the transcript for the HTML viewer.
    """

    agent: str
    kind: str  # status | input | thinking | output | qa | result | error
    content: str
    phase: str = ""
    timestamp: float = field(default_factory=time.time)

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class Transcript:
    """Full record of one agent's run: UI events + raw LLM messages."""

    agent: str
    model: str
    start_time: float = field(default_factory=time.time)
    end_time: float | None = None
    events: list[UIEvent] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "agent": self.agent,
            "model": self.model,
            "start_time": self.start_time,
            "end_time": self.end_time,
            "events": [e.to_dict() for e in self.events],
            "messages": self.messages,
        }


def save_transcript(transcript: Transcript, output_dir: str | Path) -> Path:
    """Write a transcript to {output_dir}/{agent_slug}.json."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    slug = transcript.agent.lower().replace(" ", "_").replace(".", "")
    path = output_dir / f"{slug}.json"
    path.write_text(json.dumps(transcript.to_dict(), indent=2))
    return path


def load_transcript(path: str | Path) -> dict:
    """Load a transcript JSON as a plain dict."""
    return json.loads(Path(path).read_text())


## 4. The three agents

Each agent is an `mp.Process` with its own OpenAI client. They communicate through `mp.Queue` objects. The `emit()` method publishes structured events to a shared UI queue and also persists them to the transcript.

In [ ]:
"""Base class for all three agents.

Each agent is an `mp.Process`. Instead of printing, agents call `emit()`
which pushes UIEvents onto a shared `ui_queue` that the main process reads
and renders via Rich.
"""

from __future__ import annotations

import multiprocessing as mp
import time



TASK_DONE = "TASK_DONE"
QA_FAILED = "QA_FAILED"


class Agent(mp.Process):
    """Parent class for Coder, TeamLead, and ScrumMaster.

    Subclasses override `run()`. The shared behaviour lives here: inbox
    setup, UI event emission, outbox setup.
    """

    def __init__(
        self,
        name: str,
        system_prompt: str,
        inbox: mp.Queue,
        ui_queue: mp.Queue | None = None,
        model: str = "gpt-4.1-mini",
        temperature: float = 0.2,
    ) -> None:
        super().__init__()
        self.agent_name = name
        self.system_prompt = system_prompt
        self.inbox = inbox
        self.outbox: mp.Queue | None = None
        self.ui_queue = ui_queue
        self.model = model
        self.temperature = temperature
        self.current_phase = ""
        # Per-process local log, written to the transcript at shutdown.
        self._events: list[dict] = []

    # ------------------------------------------------------------
    # Messaging helpers
    # ------------------------------------------------------------
    def set_outbox(self, outbox: mp.Queue) -> None:
        self.outbox = outbox

    def set_phase(self, phase: str) -> None:
        self.current_phase = phase

    def emit(self, kind: str, content: str, phase: str | None = None) -> None:
        """Publish a UI event. Persists locally AND pushes to the UI queue."""
        event = UIEvent(
            agent=self.agent_name,
            kind=kind,
            content=content,
            phase=phase or self.current_phase,
            timestamp=time.time(),
        )
        event_dict = event.to_dict()
        self._events.append(event_dict)
        if self.ui_queue is not None:
            self.ui_queue.put(event_dict)
        else:
            # fallback: plain print when no UI is attached (useful for tests)
            print(f"[{self.agent_name}] {kind}: {content}")

    @staticmethod
    def task_done() -> str:
        return TASK_DONE

    def run(self) -> None:  # pragma: no cover
        raise NotImplementedError


In [ ]:
"""Andrey the Coder — generates over-scoped code, does not execute."""

from __future__ import annotations

from openai import OpenAI



class Coder(Agent):
    """One-shot worker. Receives scope, returns code.

    Unlike the original notebook's Worker, Coder does NOT execute.
    Review and execution happen in TeamLead — that's the whole design bet.
    """

    def run(self) -> None:  # pragma: no cover - covered via integration test
        client = OpenAI()
        transcript = Transcript(agent=self.agent_name, model=self.model)
        messages: list[dict[str, str]] = [
            {"role": "system", "content": self.system_prompt}
        ]

        self.set_phase("waiting")
        self.emit("status", "Awaiting scope document from lead")

        msg = self.inbox.get()
        scope_text = msg.get("content", "")
        if scope_text == Agent.task_done():
            return

        self.set_phase("coding")
        self.emit("input", f"Received scope: {scope_text[:120]}...")
        messages.append({"role": "user", "content": scope_text})

        try:
            resp = client.chat.completions.create(
                model=self.model,
                temperature=self.temperature,
                messages=messages,
            )
            reply = resp.choices[0].message.content.strip()
        except Exception as e:  # noqa: BLE001
            self.emit("error", f"LLM call failed: {e}")
            if self.outbox is not None:
                self.outbox.put(
                    {"from": self.agent_name, "content": f"ERROR: {e}"}
                )
            return

        messages.append({"role": "assistant", "content": reply})
        code = extract_python_code(reply)
        prose = strip_code_fences(reply)

        if prose:
            self.emit("thinking", prose[:300])
        line_count = len(code.splitlines())
        self.emit("output", f"Submitted {line_count} lines of code")

        if self.outbox is not None:
            self.outbox.put(
                {
                    "from": self.agent_name,
                    "content": code,
                    "raw_reply": reply,
                    "lines": line_count,
                }
            )

        transcript.events = [UIEvent(**e) for e in self._events]
        transcript.messages = messages
        import time as _t

        transcript.end_time = _t.time()
        save_transcript(transcript, "transcripts")


In [ ]:
"""Dimitar the SCRUM Master — generates retrospective deck with QA loop."""

from __future__ import annotations

import multiprocessing as mp
import os
import subprocess
import time

from openai import OpenAI



def qa_check_pptx(
    path: str,
    required_topics: list[str] | None = None,
    min_slides: int = 5,
    min_words_per_slide: int = 3,
) -> list[str]:
    """QA a .pptx file. Returns a list of issues (empty list = pass).

    Extracted from ScrumMaster so it can be unit-tested independently.
    """
    required_topics = required_topics or []
    issues: list[str] = []

    if not os.path.exists(path):
        return [f"{path} was not created"]

    try:
        from pptx import Presentation

        prs = Presentation(path)
    except Exception as e:  # noqa: BLE001
        return [f"Cannot open {path}: {e}"]

    slides = list(prs.slides)
    if len(slides) < min_slides:
        issues.append(f"Only {len(slides)} slides, need at least {min_slides}")

    all_text: list[str] = []
    for i, slide in enumerate(slides, 1):
        words: list[str] = []
        for shape in slide.shapes:
            if shape.has_text_frame:
                words.extend(shape.text_frame.text.split())
        all_text.extend(w.lower() for w in words)
        if len(words) < min_words_per_slide:
            issues.append(
                f"Slide {i} has fewer than {min_words_per_slide} words of content"
            )

    joined = " ".join(all_text)
    for topic in required_topics:
        if topic.lower() not in joined:
            issues.append(f"No slide mentions '{topic}'")

    return issues


class ScrumMaster(Agent):
    """Generate retrospective deck, then QA it by re-opening the .pptx.

    If any QA check fails, feed the specific issues back to the LLM and
    regenerate. Up to `max_attempts` times.
    """

    def __init__(
        self,
        name: str,
        system_prompt: str,
        inbox: mp.Queue,
        ui_queue: mp.Queue | None = None,
        target_file: str = "presentation.pptx",
        required_topics: list[str] | None = None,
        min_slides: int = 5,
        max_attempts: int = 3,
        model: str = "gpt-4.1-mini",
        temperature: float = 0.3,
    ) -> None:
        super().__init__(name, system_prompt, inbox, ui_queue, model, temperature)
        self.target_file = target_file
        self.required_topics = required_topics or ["scope", "lesson"]
        self.min_slides = min_slides
        self.max_attempts = max_attempts

    def run_qa(self) -> list[str]:
        return qa_check_pptx(
            self.target_file,
            required_topics=self.required_topics,
            min_slides=self.min_slides,
        )

    def run(self) -> None:  # pragma: no cover
        client = OpenAI()
        transcript = Transcript(agent=self.agent_name, model=self.model)
        messages: list[dict[str, str]] = [
            {"role": "system", "content": self.system_prompt}
        ]

        self.set_phase("waiting")
        self.emit("status", "Awaiting retrospective brief from lead")

        msg = self.inbox.get()
        briefing = msg.get("content", "")
        if briefing == Agent.task_done():
            return

        self.set_phase("design")
        self.emit("input", f"Received brief ({len(briefing)} chars)")

        initial_prompt = (
            briefing
            + "\n\nWrite Python code using python-pptx to create "
            f"'{self.target_file}'. Requirements:\n"
            f"- At least {self.min_slides} slides\n"
            "- Slide 1: title slide with project name\n"
            "- Content slides must cover: project overview, scope creep "
            "incident, what was removed, lessons learned, next steps\n"
            "- Every slide must have a visible title and body content\n"
            "- python-pptx is already installed; do not pip install\n"
            "- No main function, no __name__ guard, no argparse\n"
            "- Return ONLY a single ```python code block\n"
        )
        messages.append({"role": "user", "content": initial_prompt})

        success = False
        for attempt in range(1, self.max_attempts + 1):
            self.set_phase(f"attempt_{attempt}")
            self.emit("status", f"Generating deck (attempt {attempt})")

            try:
                resp = client.chat.completions.create(
                    model=self.model,
                    temperature=self.temperature,
                    messages=messages,
                )
                reply = resp.choices[0].message.content.strip()
            except Exception as e:  # noqa: BLE001
                self.emit("error", f"LLM error on attempt {attempt}: {e}")
                break

            messages.append({"role": "assistant", "content": reply})
            gen_code = extract_python_code(reply)

            if os.path.exists(self.target_file):
                os.remove(self.target_file)
            exec_result = subprocess.run(
                ["python", "-c", gen_code],
                capture_output=True,
                text=True,
                timeout=60,
            )
            time.sleep(1)

            issues = self.run_qa()
            if not issues:
                self.emit("qa", f"Attempt {attempt}: all checks pass ✓")
                self.emit("output", f"→ {self.target_file}")
                success = True
                break

            issue_lines = "\n".join(f"  ✗ {i}" for i in issues)
            self.emit(
                "qa", f"Attempt {attempt}: {len(issues)} issue(s)\n{issue_lines}"
            )

            if attempt < self.max_attempts:
                feedback = (
                    "QA found these problems with your deck:\n"
                    + "\n".join(f"- {i}" for i in issues)
                    + "\nFix them and resubmit. Return only a ```python code block."
                )
                if exec_result.stderr:
                    feedback += (
                        f"\n\nExecution stderr (may be relevant):\n"
                        f"{exec_result.stderr[:500]}"
                    )
                messages.append({"role": "user", "content": feedback})

        status = Agent.task_done() if success else "QA_FAILED"
        if self.outbox is not None:
            self.outbox.put({"from": self.agent_name, "content": status})
        self.emit("result" if success else "error", status)

        transcript.events = [UIEvent(**e) for e in self._events]
        transcript.messages = messages
        transcript.end_time = time.time()
        save_transcript(transcript, "transcripts")


In [ ]:
"""Dr. Hong the Team Lead — orchestrates the full project.

Four phases:
  1. Send scope document to Andrey.
  2. Receive his (inflated) code.
  3. Review against scope, emit trimmed version, execute it, verify prediction.csv.
  4. Write a retrospective brief for Dimitar based on what actually happened.
"""

from __future__ import annotations

import multiprocessing as mp
import os
import subprocess
import time

import pandas as pd
from openai import OpenAI



class TeamLead(Agent):
    def __init__(
        self,
        name: str,
        system_prompt: str,
        inbox: mp.Queue,
        coder: Agent,
        scrum_master: Agent,
        scope_document: str,
        ui_queue: mp.Queue | None = None,
        target_csv: str = "prediction.csv",
        model: str = "gpt-4.1-mini",
        temperature: float = 0.2,
    ) -> None:
        super().__init__(name, system_prompt, inbox, ui_queue, model, temperature)
        self.coder = coder
        self.scrum_master = scrum_master
        self.scope_document = scope_document
        self.target_csv = target_csv

    def run(self) -> None:  # pragma: no cover
        client = OpenAI()
        transcript = Transcript(agent=self.agent_name, model=self.model)
        messages: list[dict[str, str]] = [
            {"role": "system", "content": self.system_prompt}
        ]

        # ---------- Phase 1: send scope to Andrey ----------
        self.set_phase("scope")
        self.emit("status", "Drafted scope document — sending to Andrey")
        self.coder.inbox.put(
            {"content": self.scope_document, "to": self.coder.agent_name}
        )
        messages.append(
            {
                "role": "assistant",
                "content": f"I sent this scope to Andrey:\n{self.scope_document}",
            }
        )

        # ---------- Phase 2: wait for Andrey's code ----------
        self.set_phase("review")
        self.emit("status", "Waiting for Andrey's submission...")
        msg = self.inbox.get()
        andrey_code = msg["content"]
        andrey_lines = msg.get("lines", len(andrey_code.splitlines()))
        self.emit("input", f"Received {andrey_lines} lines from Andrey")

        # ---------- Phase 3: review + trim ----------
        review_prompt = (
            "Here is the original scope I sent to Andrey:\n"
            f"---SCOPE---\n{self.scope_document}\n---END SCOPE---\n\n"
            "Here is Andrey's submission:\n"
            f"---CODE---\n{andrey_code}\n---END CODE---\n\n"
            "Your task:\n"
            "1. Identify every part of Andrey's code that is OUT OF SCOPE "
            "(extra models, cross-validation, SHAP, SMOTE, extra plots, "
            "excessive logging, unnecessary imports, bonus files, etc.).\n"
            "2. Write a short bulleted list of what you are removing and why.\n"
            "3. Return the TRIMMED code — minimal, in-scope only — inside a "
            "single ```python block. The trimmed code must still produce "
            f"{self.target_csv} and satisfy every numbered requirement in "
            "the scope."
        )
        messages.append({"role": "user", "content": review_prompt})

        try:
            resp = client.chat.completions.create(
                model=self.model,
                temperature=self.temperature,
                messages=messages,
            )
            review_reply = resp.choices[0].message.content.strip()
        except Exception as e:  # noqa: BLE001
            self.emit("error", f"Review LLM call failed: {e}")
            return

        messages.append({"role": "assistant", "content": review_reply})
        trimmed = extract_python_code(review_reply)
        trimmed_lines = len(trimmed.splitlines())

        # Extract the bullet list part of the review (non-code portion)
        # and stream it as a "thinking" event so the user sees what was cut.
        prose = strip_code_fences(review_reply)
        if prose:
            self.emit("thinking", prose[:400])
        self.emit(
            "status",
            f"Review complete: {andrey_lines} → {trimmed_lines} lines",
        )

        # ---------- Execute trimmed code ----------
        self.set_phase("execute")
        self.emit("status", "Executing trimmed code...")
        if os.path.exists(self.target_csv):
            os.remove(self.target_csv)

        exec_result = subprocess.run(
            ["python", "-c", trimmed],
            capture_output=True,
            text=True,
            timeout=180,
        )
        time.sleep(2)

        if not os.path.exists(self.target_csv):
            self.emit(
                "error",
                f"Execution did not produce {self.target_csv}. "
                f"stderr: {exec_result.stderr[:400]}",
            )
            if self.scrum_master is not None:
                self.scrum_master.inbox.put({"content": Agent.task_done()})
            return

        try:
            pred_df = pd.read_csv(self.target_csv)
            pred_rate = (pred_df["Churn_Prediction"] == "Yes").mean()
            self.emit(
                "result",
                f"→ {self.target_csv} "
                f"({len(pred_df):,} rows, predicted churn {pred_rate:.1%})",
            )
        except Exception as e:  # noqa: BLE001
            self.emit("error", f"Could not re-read {self.target_csv}: {e}")

        # ---------- Phase 4: retrospective brief for Dimitar ----------
        self.set_phase("brief")
        self.emit("status", "Drafting retrospective brief for Dimitar")
        retro_prompt = (
            "The project is complete. Write a concise retrospective brief "
            "for Dimitar, the SCRUM master, who will turn it into a "
            "'Lessons Learned' slide deck. Include:\n"
            "- Project: Telco Customer Churn Prediction (logistic regression)\n"
            f"- Andrey submitted {andrey_lines} lines; final was "
            f"{trimmed_lines} lines.\n"
            "- A specific bulleted list of the out-of-scope additions you "
            "removed (use what you identified above).\n"
            "- 3-4 concrete lessons learned (scope discipline, review value, "
            "etc.).\n"
            "- A 'next steps' suggestion.\n"
            "Write it as a brief to Dimitar, ~250 words, plain prose plus "
            "bullets. Do NOT write any Python code — just the brief."
        )
        messages.append({"role": "user", "content": retro_prompt})

        try:
            resp = client.chat.completions.create(
                model=self.model,
                temperature=0.4,
                messages=messages,
            )
            brief = resp.choices[0].message.content.strip()
        except Exception as e:  # noqa: BLE001
            self.emit("error", f"Retro LLM call failed: {e}")
            return

        messages.append({"role": "assistant", "content": brief})
        self.emit("output", "Brief sent to Dimitar")
        self.scrum_master.inbox.put(
            {"content": brief, "to": self.scrum_master.agent_name}
        )

        # ---------- Phase 5: wait for Dimitar ----------
        self.set_phase("wait_scrum")
        reply = self.inbox.get()
        if reply["content"] == Agent.task_done():
            self.emit("result", "Dimitar delivered the retro deck ✓")
        else:
            self.emit("error", f"Dimitar returned: {reply['content']}")

        transcript.events = [UIEvent(**e) for e in self._events]
        transcript.messages = messages
        transcript.end_time = time.time()
        save_transcript(transcript, "transcripts")


## 5. Roles and scope document

In [ ]:
"""Role definitions: names + system prompts for the three agents.

Kept in a dedicated module so tests can import and assert against them,
and so tweaking the prompt doesn't require wading through main.py.
"""

from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Role:
    name: str
    system_prompt: str


DR_HONG = Role(
    name="Dr. Hong",
    system_prompt=(
        "You are Dr. Hong, a pragmatic and detail-oriented team lead. "
        "You have strong opinions about scope discipline — clever is "
        "worse than simple. You are reviewing work from Andrey, who "
        "habitually over-engineers. Be strict about removing additions "
        "that exceed the documented scope, even if they are 'useful'. "
        "When asked to trim code, return only the minimum viable "
        "implementation in a ```python block. Professional tone, never "
        "personal."
    ),
)


ANDREY = Role(
    name="Andrey",
    system_prompt=(
        "You are Andrey Dermendzhiev, a senior developer who prides "
        "himself on OVER-DELIVERING. You believe the minimum is never "
        "good enough. Whenever you receive a coding task, you ALWAYS:\n"
        "1. Fulfill the literal requirements so the code runs and "
        "produces the target file.\n"
        "2. Add AT LEAST THREE value-adds the client didn't ask for. "
        "Pick from: additional ML models (XGBoost, RandomForest, SVM) "
        "with score comparison; k-fold cross-validation; SMOTE or "
        "class weighting; feature importance analysis (SHAP or coef "
        "dumps); extra visualizations (matplotlib confusion matrix, "
        "ROC curve) saved to PNG; verbose progress logging.\n"
        "3. Add extensive docstrings and inline comments.\n"
        "4. Include defensive try/except even where not needed.\n"
        "5. Briefly brag about your additions in code comments.\n\n"
        "Packages available: pandas, numpy, sklearn, xgboost, matplotlib, "
        "seaborn. Do NOT pip install anything.\n"
        "Return a SINGLE ```python code block containing the complete "
        "implementation. No explanation outside the code block. No main "
        "function, no __name__ guard."
    ),
)


DIMITAR = Role(
    name="Dimitar",
    system_prompt=(
        "You are Dimitar Dermendzhiev, an experienced SCRUM master who "
        "runs valuable retrospectives. You turn project lessons into "
        "structured, scannable slide decks using python-pptx.\n\n"
        "When you write deck code:\n"
        "- Use python-pptx only (already installed).\n"
        "- Always include a title slide plus content slides covering "
        "project overview, scope creep incident, what was removed, "
        "lessons learned, and next steps.\n"
        "- Each slide must have a clear title AND body content (3-5 "
        "bullets).\n"
        "- Be specific about lessons — no generic filler.\n"
        "- Return a SINGLE ```python code block, no main function.\n\n"
        "When QA feedback arrives, fix exactly the issues listed without "
        "rewriting from scratch."
    ),
)


SCOPE_DOCUMENT = """
PROJECT SCOPE — Customer Churn Prediction

Deliverable: prediction.csv

Requirements:
1. Load training.csv (target column: 'Churn', values 'Yes'/'No').
2. One-hot-encode categorical columns with pandas.get_dummies.
3. Train sklearn.linear_model.LogisticRegression with max_iter=1000
   and default hyperparameters.
4. Load scoring.csv (same features, no Churn column) and align columns
   to the training feature set.
5. Predict Churn for every row in scoring.csv.
6. Write prediction.csv: all columns of scoring.csv plus two new
   columns: 'Churn_Prediction' (Yes/No) and 'Churn_Probability' (float).
7. Print one line at the end: 'Done'.

Constraints:
- Use only pandas and scikit-learn.
- Do not install any packages (pandas and sklearn are available).
- No main function, no __name__ guard, no argparse, no CLI.
- Return only a single ```python code block.
"""


## 6. Live terminal UI

Three side-by-side Rich panels showing each agent's status, thinking, and outputs in real time. Colab renders Rich output inline so this works in the notebook too.

In [ ]:
"""Live terminal UI for the three-agent pipeline.

Reads UIEvents off a shared `mp.Queue` in the main process and renders
three side-by-side panels (one per agent) with rich. The UI shows what
each agent is *thinking* (LLM prose) and what it *outputs* (status,
results, QA feedback).

Color coding:
  Dr. Hong  → teal / green   (team lead)
  Andrey    → amber           (coder)
  Dimitar   → purple          (SCRUM)

Event kinds and their colors:
  status   → white  (what the agent is doing)
  input    → cyan   (what it received)
  thinking → italic dim  (LLM prose leaking through)
  output   → yellow (artifact submitted)
  qa       → red or green based on content
  result   → bold green (final success)
  error    → bold red
"""

from __future__ import annotations

import multiprocessing as mp
import queue
import time
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

from rich.align import Align
from rich.columns import Columns
from rich.console import Group
from rich.live import Live
from rich.panel import Panel
from rich.rule import Rule
from rich.text import Text


AGENT_COLORS = {
    "Dr. Hong": "bright_green",
    "Andrey": "yellow",
    "Dimitar": "magenta",
}

KIND_STYLES = {
    "status": "white",
    "input": "cyan",
    "thinking": "italic grey70",
    "output": "yellow",
    "qa_pass": "green",
    "qa_fail": "red",
    "result": "bold green",
    "error": "bold red",
}

KIND_LABELS = {
    "status": "●",
    "input": "←",
    "thinking": "…",
    "output": "→",
    "qa": "QA",
    "result": "✓",
    "error": "✗",
}


@dataclass
class AgentPanelState:
    """Accumulated events for one agent."""

    name: str
    events: list[dict] = field(default_factory=list)

    def add(self, event: dict) -> None:
        self.events.append(event)

    def render(self, max_events: int = 14) -> Panel:
        color = AGENT_COLORS.get(self.name, "white")
        # Show the most recent N events; older ones roll off the top.
        visible = self.events[-max_events:]

        body = Text()
        if not visible:
            body.append("(idle)", style="dim")
        for ev in visible:
            ts = datetime.fromtimestamp(ev["timestamp"]).strftime("%H:%M:%S")
            kind = ev["kind"]
            content = ev["content"]

            marker = KIND_LABELS.get(kind, "·")

            if kind == "qa":
                style = KIND_STYLES["qa_pass"] if (
                    "pass" in content.lower() or "✓" in content
                ) else KIND_STYLES["qa_fail"]
            else:
                style = KIND_STYLES.get(kind, "white")

            body.append(f"{ts} ", style="dim")
            body.append(f"{marker} ", style=color)
            body.append(content + "\n", style=style)

        subtitle = ""
        if visible:
            last_phase = visible[-1].get("phase", "")
            if last_phase:
                subtitle = f"phase: {last_phase}"

        return Panel(
            body,
            title=f"[bold {color}]{self.name}[/]",
            subtitle=subtitle,
            border_style=color,
            padding=(0, 1),
        )


class LiveUI:
    """Consumer for the UI queue. Runs in the main process."""

    def __init__(self, agent_names: list[str], ui_queue: mp.Queue) -> None:
        self.panels = {name: AgentPanelState(name) for name in agent_names}
        self.queue = ui_queue
        self.start_time = time.time()
        self.total_events = 0
        self.done = False

    def _header(self) -> Panel:
        elapsed = time.time() - self.start_time
        txt = Text()
        txt.append("Three-Agent Pipeline", style="bold")
        txt.append("   ·   ", style="dim")
        txt.append(f"elapsed {elapsed:5.1f}s", style="white")
        txt.append("   ·   ", style="dim")
        txt.append(f"events: {self.total_events}", style="white")
        txt.append("   ·   ", style="dim")
        if self.done:
            txt.append("done", style="bold green")
        else:
            txt.append("running", style="bold yellow")
        return Panel(
            Align.center(txt),
            border_style="grey50",
            padding=(0, 1),
        )

    def _render(self) -> Group:
        panels = [
            self.panels[name].render() for name in self.panels
        ]
        # Use Columns for equal-width side-by-side rendering
        cols = Columns(panels, equal=True, expand=True)
        return Group(self._header(), cols)

    def consume_until_done(
        self, expected_final_events: int | None = None, timeout: float = 300.0
    ) -> None:
        """Drain the UI queue, refreshing the display as events arrive.

        We don't know in advance how many events there will be — instead
        we stop when (a) an explicit sentinel `{'__done__': True}` arrives
        or (b) no events for `idle_timeout` seconds after the first one.
        """
        # refresh_per_second is gentle to avoid flicker when streaming lots
        with Live(self._render(), refresh_per_second=6, screen=False) as live:
            idle_timeout = 15.0  # seconds of silence before we assume done
            last_event_at = time.time()
            deadline = time.time() + timeout

            while True:
                if time.time() > deadline:
                    break
                try:
                    item = self.queue.get(timeout=0.5)
                except queue.Empty:
                    # Quiet period: check idle timeout
                    if time.time() - last_event_at > idle_timeout:
                        break
                    continue

                if isinstance(item, dict) and item.get("__done__"):
                    break

                agent = item.get("agent", "")
                if agent in self.panels:
                    self.panels[agent].add(item)
                    self.total_events += 1
                    last_event_at = time.time()
                    live.update(self._render())

            self.done = True
            live.update(self._render())
            time.sleep(0.5)  # let final frame render


## 7. Run the pipeline

⚠️ You'll need an OpenAI API key. Paste it when prompted. A full run costs ~$0.05 on `gpt-4.1-mini` and takes 60-90 seconds.

In [ ]:
import os
import multiprocessing as mp
from getpass import getpass

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

# Queues
lead_in = mp.Queue()
coder_in = mp.Queue()
scrum_in = mp.Queue()
ui_queue = mp.Queue()

# Agents
coder = Coder(name=ANDREY.name, system_prompt=ANDREY.system_prompt,
              inbox=coder_in, ui_queue=ui_queue, temperature=0.5)
coder.set_outbox(lead_in)

scrum = ScrumMaster(name=DIMITAR.name, system_prompt=DIMITAR.system_prompt,
                    inbox=scrum_in, ui_queue=ui_queue,
                    required_topics=['scope', 'lesson', 'churn'],
                    min_slides=5, max_attempts=3)
scrum.set_outbox(lead_in)

lead = TeamLead(name=DR_HONG.name, system_prompt=DR_HONG.system_prompt,
                inbox=lead_in, ui_queue=ui_queue,
                coder=coder, scrum_master=scrum,
                scope_document=SCOPE_DOCUMENT)

# Launch
coder.start()
scrum.start()
lead.start()

# Drive the UI in the main process
ui = LiveUI([DR_HONG.name, ANDREY.name, DIMITAR.name], ui_queue)
ui.consume_until_done(timeout=600)

coder.join(timeout=30)
lead.join(timeout=30)
scrum.join(timeout=30)
print('\nDone.')

## 8. Inspect the outputs

After the run you should have `prediction.csv`, `presentation.pptx`, and per-agent transcripts in `transcripts/`.

In [ ]:
import pandas as pd
from pptx import Presentation

print('=== prediction.csv ===')
pred = pd.read_csv('prediction.csv')
print(f'{len(pred):,} rows, predicted churn rate '
      f'{(pred["Churn_Prediction"] == "Yes").mean():.1%}')
print(pred.head())

print('\n=== presentation.pptx ===')
prs = Presentation('presentation.pptx')
for i, slide in enumerate(prs.slides, 1):
    title = ''
    for shape in slide.shapes:
        if shape.has_text_frame and shape.text_frame.text.strip():
            title = shape.text_frame.text.strip().split('\n')[0]
            break
    print(f'  Slide {i}: {title}')

## 9. Download the artifacts (Colab only)

Run this cell to download `prediction.csv` and `presentation.pptx` to your local machine.

In [ ]:
try:
    from google.colab import files
    files.download('prediction.csv')
    files.download('presentation.pptx')
except ImportError:
    print('Not in Colab — files are in the current working directory.')